# Build appeal-chain relevant court manifest

This notebook builds the **procedure-level Court manifest relevant to the appeal-chain analysis**.

The input `infocuria_manual_citations.xlsx` is citation-level: the same Court document can appear many times because it can contain several Commission references. The goal here is not yet to construct appeal edges. It is simply to identify the InfoCuria procedures represented by those manually curated Court documents and retain their metadata.

## Mapping logic

1. Collapse `infocuria_manual_citations.xlsx` to **one row per `document_id`**.
2. Map each document to the case metadata manifest:
   - primary mapping: `manual.publishedId == metadata.procedureId`;
   - conservative fallback: `affId`, **only when that `affId` corresponds to exactly one `procedureId`**.
3. Keep one metadata row per matched `procedureId`.
4. Add provenance showing which source Court documents caused the procedure to be retained.
5. Save `appeal_chain_relevant_court_manifest.xlsx`.

### Important identifier note

The case metadata column `relevant_document_ids` contains InfoCuria `logicDocId`-style values such as `id_46950`. These are **not** the same identifier family as the manually curated workbook's numeric `document_id`, so they are not used as the join key.

In the current citation output, `publishedId` is already the full InfoCuria procedure identifier, for example:

`C/0008/66/00000000RD/01/P/01`

and therefore maps directly to the metadata manifest's `procedureId`.


In [3]:
from __future__ import annotations

from pathlib import Path
import os
import re

import pandas as pd
from IPython.display import display


# ---------------------------------------------------------------------
# Project paths
# ---------------------------------------------------------------------

NOTEBOOK_DIR = Path.cwd().resolve()


def find_project_root(start: Path) -> Path:
    """Find the ECCJEU project root from the current notebook location."""
    for candidate in [start, *start.parents]:
        if (candidate / "input").exists() and (candidate / "output").exists():
            return candidate
        if (candidate / "data").exists() and (candidate / "output").exists():
            return candidate
    return start


PROJECT_ROOT = find_project_root(NOTEBOOK_DIR)

MANUAL_CITATIONS_PATH = Path(
    os.getenv(
        "MANUAL_CITATIONS_PATH",
        str(PROJECT_ROOT / "input" / "infocuria_manual_citations.xlsx"),
    )
).expanduser().resolve()

CASE_METADATA_XLSX = (
    PROJECT_ROOT
    / "output"
    / "court_infocuria"
    / "excel"
    / "infocuria_case_metadata_manifest.xlsx"
)

CASE_METADATA_PARQUET = (
    PROJECT_ROOT
    / "output"
    / "court_infocuria"
    / "parquet"
    / "infocuria_case_metadata_manifest.parquet"
)

# Optional override if the manifest lives somewhere else.
CASE_METADATA_OVERRIDE = os.getenv("CASE_METADATA_PATH", "").strip()

if CASE_METADATA_OVERRIDE:
    CASE_METADATA_PATH = Path(
        CASE_METADATA_OVERRIDE
    ).expanduser().resolve()
elif CASE_METADATA_PARQUET.exists():
    # COR2 writes this alongside the Excel manifest. Prefer it for speed.
    CASE_METADATA_PATH = CASE_METADATA_PARQUET.resolve()
else:
    CASE_METADATA_PATH = CASE_METADATA_XLSX.resolve()

OUTPUT_DIR = (
    PROJECT_ROOT
    / "output"
    / "court_infocuria"
    / "excel"
)

OUTPUT_PATH = Path(
    os.getenv(
        "APPEAL_CHAIN_RELEVANT_MANIFEST_PATH",
        str(
            OUTPUT_DIR
            / "appeal_chain_relevant_court_manifest.xlsx"
        ),
    )
).expanduser().resolve()

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Manual citations:", MANUAL_CITATIONS_PATH)
print("Case metadata:", CASE_METADATA_PATH)
print("Output:", OUTPUT_PATH)


PROJECT_ROOT: /home/edik/projects/eccjeu
Manual citations: /home/edik/projects/eccjeu/input/infocuria_manual_citations.xlsx
Case metadata: /home/edik/projects/eccjeu/output/court_infocuria/parquet/infocuria_case_metadata_manifest.parquet
Output: /home/edik/projects/eccjeu/output/court_infocuria/excel/appeal_chain_relevant_court_manifest.xlsx


## 1. Load inputs

The notebook prefers the Parquet version of the case metadata manifest when COR2 has produced it, because it is much faster to read. If only the Excel manifest exists, it uses that instead.


In [4]:
# ---------------------------------------------------------------------
# Input readers
# ---------------------------------------------------------------------

def read_table(path: Path, *, sheet_name=None) -> pd.DataFrame:
    suffix = path.suffix.lower()

    if suffix == ".parquet":
        return pd.read_parquet(path)

    if suffix in {".xlsx", ".xls"}:
        if sheet_name is None:
            return pd.read_excel(path)
        return pd.read_excel(
            path,
            sheet_name=sheet_name,
        )

    if suffix == ".jsonl":
        return pd.read_json(
            path,
            orient="records",
            lines=True,
        )

    if suffix == ".csv":
        return pd.read_csv(
            path,
            low_memory=False,
        )

    raise ValueError(
        f"Unsupported input format: {path}"
    )


if not MANUAL_CITATIONS_PATH.exists():
    raise FileNotFoundError(
        f"Manual citation workbook not found: "
        f"{MANUAL_CITATIONS_PATH}"
    )

if not CASE_METADATA_PATH.exists():
    raise FileNotFoundError(
        "Case metadata manifest not found. Expected either:\n"
        f"  - {CASE_METADATA_PARQUET}\n"
        f"  - {CASE_METADATA_XLSX}\n"
        "or set CASE_METADATA_PATH explicitly."
    )

manual = read_table(
    MANUAL_CITATIONS_PATH,
    sheet_name="citations_mapped",
)

metadata = read_table(
    CASE_METADATA_PATH,
)

manual.columns = [
    str(c).strip()
    for c in manual.columns
]

metadata.columns = [
    str(c).strip()
    for c in metadata.columns
]

required_manual = {
    "document_id",
    "publishedId",
    "affId",
}

required_metadata = {
    "procedureId",
    "affId",
}

missing_manual = sorted(
    required_manual - set(manual.columns)
)

missing_metadata = sorted(
    required_metadata - set(metadata.columns)
)

if missing_manual:
    raise ValueError(
        "Manual citation workbook is missing "
        f"required columns: {missing_manual}"
    )

if missing_metadata:
    raise ValueError(
        "Case metadata manifest is missing "
        f"required columns: {missing_metadata}"
    )

print(f"Manual citation rows: {len(manual):,}")
print(f"Metadata rows: {len(metadata):,}")


Manual citation rows: 14,529
Metadata rows: 56,828


## 2. Normalize identifiers and collapse to one row per `document_id`

A single Court document can have several citation rows. Before mapping, we verify that all rows belonging to one `document_id` agree on the Court identifiers.


In [5]:
# ---------------------------------------------------------------------
# Identifier helpers
# ---------------------------------------------------------------------

def clean_text(value) -> str:
    if value is None:
        return ""

    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass

    text = str(value).strip()

    if text.lower() in {
        "",
        "nan",
        "none",
        "<na>",
    }:
        return ""

    return text


def clean_document_id(value) -> str:
    """Normalize IDs without destroying suffixes such as __dup001."""
    if value is None:
        return ""

    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass

    if isinstance(value, int):
        return str(value)

    if isinstance(value, float) and value.is_integer():
        return str(int(value))

    text = str(value).strip()

    # Excel can sometimes turn an integer ID into '123.0'.
    if re.fullmatch(r"\d+\.0", text):
        return text[:-2]

    return text


def ordered_unique_nonempty(values) -> list[str]:
    out = []
    seen = set()

    for value in values:
        text = clean_text(value)

        if text and text not in seen:
            seen.add(text)
            out.append(text)

    return out


def first_nonempty(series: pd.Series) -> str:
    values = ordered_unique_nonempty(series)
    return values[0] if values else ""


def pipe_join(values) -> str:
    return "|".join(
        ordered_unique_nonempty(values)
    )


# ---------------------------------------------------------------------
# Normalize join identifiers
# ---------------------------------------------------------------------

manual_work = manual.copy()

manual_work["document_id"] = (
    manual_work["document_id"]
    .map(clean_document_id)
)

manual_work["publishedId"] = (
    manual_work["publishedId"]
    .map(clean_text)
)

manual_work["affId"] = (
    manual_work["affId"]
    .map(clean_text)
)

if "court_case" in manual_work.columns:
    manual_work["court_case"] = (
        manual_work["court_case"]
        .map(clean_text)
    )

metadata["procedureId"] = (
    metadata["procedureId"]
    .map(clean_text)
)

metadata["affId"] = (
    metadata["affId"]
    .map(clean_text)
)


# ---------------------------------------------------------------------
# Ensure one document does not point to conflicting case identifiers
# ---------------------------------------------------------------------

identifier_columns = [
    c
    for c in [
        "publishedId",
        "affId",
        "court_case",
    ]
    if c in manual_work.columns
]

conflict_records = []

for document_id, group in manual_work.groupby(
    "document_id",
    dropna=False,
):
    if not document_id:
        continue

    for column in identifier_columns:
        values = ordered_unique_nonempty(
            group[column]
        )

        if len(values) > 1:
            conflict_records.append({
                "document_id": document_id,
                "column": column,
                "values": " | ".join(values),
                "n_values": len(values),
            })

identifier_conflicts = pd.DataFrame(
    conflict_records
)

if not identifier_conflicts.empty:
    display(
        identifier_conflicts.head(50)
    )

    raise ValueError(
        "At least one document_id has conflicting "
        "procedure/case identifiers. Resolve these "
        "rows before building the manifest."
    )


# ---------------------------------------------------------------------
# Collapse citations -> unique Court documents
# ---------------------------------------------------------------------

document_agg = {
    "publishedId": first_nonempty,
    "affId": first_nonempty,
}

if "court_case" in manual_work.columns:
    document_agg["court_case"] = first_nonempty

manual_documents = (
    manual_work[
        manual_work["document_id"].ne("")
    ]
    .groupby(
        "document_id",
        as_index=False,
        sort=False,
    )
    .agg(document_agg)
)

print(
    f"Unique manually relevant Court documents: "
    f"{len(manual_documents):,}"
)

display(
    manual_documents.head(10)
)


Unique manually relevant Court documents: 6,327


,document_id,publishedId,affId,court_case
0,614173,T/0348/94/00000000RD/01/P/01,T/0348/94/00000000RD/01,T-348/94
1,646879,T/0161/05/00000000RD/01/P/01,T/0161/05/00000000RD/01,T-161/05
2,640505,T/0101/05/00000000RD/01/P/01,T/0101/05/00000000RD/01,T-101/05
3,640506,T/0111/05/00000000RD/01/P/01,T/0111/05/00000000RD/01,T-111/05
4,646878,T/0174/05/00000000RD/01/P/01,T/0174/05/00000000RD/01,T-174/05
5,638208,T/0410/03/00000000RD/01/P/01,T/0410/03/00000000RD/01,T-410/03
6,646880,T/0168/05/00000000RD/01/P/01,T/0168/05/00000000RD/01,T-168/05
7,614726,T/0305/94/00000000RD/01/P/01,T/0305/94/00000000RD/01,T-305/94
8,614727,T/0306/94/00000000RD/01/P/01,T/0306/94/00000000RD/01,T-306/94
9,614728,T/0307/94/00000000RD/01/P/01,T/0307/94/00000000RD/01,T-307/94


## 3. Map documents to `procedureId`

The primary mapping is exact:

`manual.publishedId → metadata.procedureId`

If that fails, an `affId` fallback is permitted only when the `affId` identifies exactly one procedure in the metadata manifest.


In [6]:
# ---------------------------------------------------------------------
# Validate metadata procedure key
# ---------------------------------------------------------------------

metadata_nonempty = metadata[
    metadata["procedureId"].ne("")
].copy()

duplicate_procedure_rows = metadata_nonempty[
    metadata_nonempty[
        "procedureId"
    ].duplicated(keep=False)
].copy()

if not duplicate_procedure_rows.empty:
    preview_columns = [
        c
        for c in [
            "case_number_clean",
            "affId",
            "procedureId",
            "publishedId",
        ]
        if c in duplicate_procedure_rows.columns
    ]

    display(
        duplicate_procedure_rows[
            preview_columns
        ].head(50)
    )

    raise ValueError(
        "The case metadata manifest contains duplicate "
        "procedureId rows. The output requires one row "
        "per procedureId."
    )

procedure_ids = set(
    metadata_nonempty["procedureId"]
)


# ---------------------------------------------------------------------
# Safe affId -> procedureId fallback
# ---------------------------------------------------------------------

affid_to_procedures = (
    metadata_nonempty[
        metadata_nonempty["affId"].ne("")
    ]
    .groupby("affId")["procedureId"]
    .agg(ordered_unique_nonempty)
)

unique_affid_lookup = {
    aff_id: procedures[0]
    for aff_id, procedures
    in affid_to_procedures.items()
    if len(procedures) == 1
}


# ---------------------------------------------------------------------
# Map each Court document
# ---------------------------------------------------------------------

document_mapping = (
    manual_documents.copy()
)

document_mapping["procedureId"] = ""
document_mapping["mapping_method"] = (
    "unmatched"
)

exact_mask = (
    document_mapping["publishedId"]
    .isin(procedure_ids)
)

document_mapping.loc[
    exact_mask,
    "procedureId",
] = document_mapping.loc[
    exact_mask,
    "publishedId",
]

document_mapping.loc[
    exact_mask,
    "mapping_method",
] = "publishedId_exact"

fallback_mask = (
    document_mapping["procedureId"].eq("")
    & document_mapping["affId"].ne("")
    & document_mapping["affId"].isin(
        unique_affid_lookup
    )
)

document_mapping.loc[
    fallback_mask,
    "procedureId",
] = (
    document_mapping.loc[
        fallback_mask,
        "affId",
    ]
    .map(unique_affid_lookup)
)

document_mapping.loc[
    fallback_mask,
    "mapping_method",
] = "affId_unique_fallback"


matched_documents = document_mapping[
    document_mapping["procedureId"].ne("")
].copy()

unmatched_documents = document_mapping[
    document_mapping["procedureId"].eq("")
].copy()


print("Mapping methods:")
print(
    document_mapping[
        "mapping_method"
    ]
    .value_counts(dropna=False)
    .to_string()
)

print(
    f"\nMatched documents: "
    f"{len(matched_documents):,}"
)

print(
    f"Unmatched documents: "
    f"{len(unmatched_documents):,}"
)

print(
    "Distinct matched procedures: "
    f"{matched_documents['procedureId'].nunique():,}"
)

if not unmatched_documents.empty:
    print("\nUnmatched document audit:")
    display(
        unmatched_documents.head(50)
    )


Mapping methods:
mapping_method
publishedId_exact    6321
unmatched               6

Matched documents: 6,321
Unmatched documents: 6
Distinct matched procedures: 4,662

Unmatched document audit:


,document_id,publishedId,affId,court_case,procedureId,mapping_method
3271,1827041__dup001,,,,,unmatched
3272,1827041__dup002,,,,,unmatched
5016,1961150__dup002,,,,,unmatched
5491,1686441__dup001,,,,,unmatched
5492,1686441__dup002,,,,,unmatched
5743,1961150__dup001,,,,,unmatched


## 4. Build `appeal_chain_relevant_court_manifest`

Every matched `procedureId` is retained once from the original case metadata manifest.

All existing metadata columns are preserved, and four provenance fields are appended:

- `appeal_chain_source_document_count`
- `appeal_chain_source_document_ids`
- `appeal_chain_source_court_cases`
- `appeal_chain_mapping_methods`


In [7]:
# ---------------------------------------------------------------------
# Aggregate source-document provenance by procedure
# ---------------------------------------------------------------------

named_agg = {
    "appeal_chain_source_document_count": (
        "document_id",
        "nunique",
    ),
    "appeal_chain_source_document_ids": (
        "document_id",
        pipe_join,
    ),
    "appeal_chain_mapping_methods": (
        "mapping_method",
        pipe_join,
    ),
}

if "court_case" in matched_documents.columns:
    named_agg[
        "appeal_chain_source_court_cases"
    ] = (
        "court_case",
        pipe_join,
    )

procedure_provenance = (
    matched_documents
    .groupby(
        "procedureId",
        as_index=False,
        sort=False,
    )
    .agg(**named_agg)
)


# ---------------------------------------------------------------------
# Filter full case metadata to relevant procedures
# ---------------------------------------------------------------------

appeal_chain_relevant_court_manifest = (
    metadata_nonempty.merge(
        procedure_provenance,
        on="procedureId",
        how="inner",
        validate="one_to_one",
    )
)

print(
    "Appeal-chain relevant procedure rows: "
    f"{len(appeal_chain_relevant_court_manifest):,}"
)

preview_columns = [
    c
    for c in [
        "case_number_clean",
        "case_name_en",
        "affId",
        "procedureId",
        "nature",
        "procType",
        "appeal_chain_source_document_count",
        "appeal_chain_source_document_ids",
        "appeal_chain_source_court_cases",
        "appeal_chain_mapping_methods",
    ]
    if c
    in appeal_chain_relevant_court_manifest.columns
]

display(
    appeal_chain_relevant_court_manifest[
        preview_columns
    ].head(20)
)


Appeal-chain relevant procedure rows: 4,662


,case_number_clean,case_name_en,affId,procedureId,nature,procType,appeal_chain_source_document_count,appeal_chain_source_document_ids,appeal_chain_source_court_cases,appeal_chain_mapping_methods
0,C-1/00,Commission v France,C/0001/00/00000000RD/01,C/0001/00/00000000RD/01/P/01,Recours direct,P,1,616977,C-1/00,publishedId_exact
1,C-1/19 P,JPMorgan Chase and Others v Commission,C/0001/19/0000000PVR/01,C/0001/19/0000000PVR/01/P/01,Pourvoi sur référé,P,1,2467790,C-1/19 P(R),publishedId_exact
2,C-1/19 P,JPMorgan Chase and Others v Commission,C/0001/19/0000000PVR/01,C/0001/19/0000000PVR/01/R/01,Pourvoi sur référé,R,1,2445341,C-1/19 P(R)-R,publishedId_exact
3,C-1/83,IFG v Freistaat Bayern,C/0001/83/00000000RP/01,C/0001/83/00000000RP/01/P/01,Procédure préjudicielle,P,1,1225637,C-1/83,publishedId_exact
4,C-1/84 R,Ilford v Commission,C/0001/84/00000000RD/01,C/0001/84/00000000RD/01/R/01,Recours direct,R,1,1226224,C-1/84 R,publishedId_exact
5,C-1/98 P,Corus UK v Commission,C/0001/98/00000000PV/01,C/0001/98/00000000PV/01/P/01,Pourvoi,P,1,615928,C-1/98 P,publishedId_exact
6,C-2/01 P,BAI and Commission v Bayer,C/0002/01/00000000PV/01,C/0002/01/00000000PV/01/P/01,Pourvoi,P,1,618730,C-2/01 P,publishedId_exact
7,C-2/24 P,Teva Pharmaceutical Industries and Cephalon v ...,C/0002/24/00000000PV/01,C/0002/24/00000000PV/01/P/01,Pourvoi,P,1,3363294,C-2/24 P,publishedId_exact
8,C-2/75,Einfuhr- und Vorratsstelle für Getreide und Fu...,C/0002/75/00000000RP/01,C/0002/75/00000000RP/01/P/01,Procédure préjudicielle,P,2,1222289|1222276,C-2/75,publishedId_exact
9,C-3/00,Denmark v Commission,C/0003/00/00000000RD/01,C/0003/00/00000000RD/01/P/01,Recours direct,P,1,618095,C-3/00,publishedId_exact


## 5. QA

The output should be strictly procedure-level. These assertions make the notebook fail if the mapping or metadata structure changes unexpectedly.


In [8]:
# ---------------------------------------------------------------------
# QA
# ---------------------------------------------------------------------

assert (
    appeal_chain_relevant_court_manifest[
        "procedureId"
    ]
    .ne("")
    .all()
)

assert (
    appeal_chain_relevant_court_manifest[
        "procedureId"
    ]
    .is_unique
)

assert matched_documents[
    "document_id"
].is_unique

matched_procedure_ids = set(
    matched_documents["procedureId"]
)

manifest_procedure_ids = set(
    appeal_chain_relevant_court_manifest[
        "procedureId"
    ]
)

assert (
    matched_procedure_ids
    == manifest_procedure_ids
)

provenance_document_count = int(
    appeal_chain_relevant_court_manifest[
        "appeal_chain_source_document_count"
    ].sum()
)

assert (
    provenance_document_count
    == len(matched_documents)
)

print("QA passed.")
print(
    f"  Unique input documents: "
    f"{len(manual_documents):,}"
)
print(
    f"  Matched documents:      "
    f"{len(matched_documents):,}"
)
print(
    f"  Unmatched documents:    "
    f"{len(unmatched_documents):,}"
)
print(
    f"  Relevant procedures:    "
    f"{len(appeal_chain_relevant_court_manifest):,}"
)


QA passed.
  Unique input documents: 6,327
  Matched documents:      6,321
  Unmatched documents:    6
  Relevant procedures:    4,662


## 6. Save output

The first sheet is the actual downstream manifest. The additional sheets make the filtering auditable without creating separate files.


In [9]:
# ---------------------------------------------------------------------
# Save
# ---------------------------------------------------------------------

with pd.ExcelWriter(
    OUTPUT_PATH,
    engine="openpyxl",
) as writer:

    appeal_chain_relevant_court_manifest.to_excel(
        writer,
        sheet_name="manifest",
        index=False,
    )

    document_mapping.to_excel(
        writer,
        sheet_name="document_mapping",
        index=False,
    )

    unmatched_documents.to_excel(
        writer,
        sheet_name="unmatched_documents",
        index=False,
    )

    for sheet_name in [
        "manifest",
        "document_mapping",
        "unmatched_documents",
    ]:
        worksheet = writer.book[sheet_name]
        worksheet.freeze_panes = "A2"
        worksheet.auto_filter.ref = (
            worksheet.dimensions
        )

print("Saved:", OUTPUT_PATH)


Saved: /home/edik/projects/eccjeu/output/court_infocuria/excel/appeal_chain_relevant_court_manifest.xlsx


## Result

`appeal_chain_relevant_court_manifest.xlsx` is now the procedure-level Court universe for the appeal-chain stage.

The next notebook can work only on this reduced procedure set and use the manually curated direct-appeal classifications to construct the actual Commission → General Court → Court of Justice chain.
